In [45]:
# --- imports ---
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr

In [46]:
# --- model (identical to evaluate_saved_models.py) ---
class CrossEmbeddingAttention(nn.Module):
    def __init__(self, input_dim=3840, reduced_dim=512, num_scores=5, embed_dim=768):
        super().__init__()
        self.embed_dim = embed_dim; self.num_scores = num_scores; self.reduced_dim = reduced_dim
        self.embedding_projections = nn.ModuleList([
            nn.Sequential(nn.Linear(embed_dim, reduced_dim), nn.ReLU(), nn.Dropout(0.4))
            for _ in range(num_scores)])
        self.cross_attention = nn.MultiheadAttention(embed_dim=reduced_dim, num_heads=8,
                                                     dropout=0.3, batch_first=True)
        self.attention_norm = nn.LayerNorm(reduced_dim)
        self.final_attention = nn.Sequential(nn.Linear(num_scores, 32), nn.ReLU(),
                                             nn.Dropout(0.4), nn.Linear(32, num_scores))
    def forward(self, x):
        b = x.size(0)
        emb = x.view(b, self.num_scores, self.embed_dim)
        proj = torch.stack([p(emb[:, i, :]) for i, p in enumerate(self.embedding_projections)], dim=1)
        att, _ = self.cross_attention(proj, proj, proj)
        att = self.attention_norm(att + proj)
        att_t = att.transpose(1, 2)
        w = F.softmax(self.final_attention(att_t), dim=-1)
        return (w * att_t).sum(dim=-1)

class DDGPredictor(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=768, dropout_rate=0.35, num_hidden=3):
        super().__init__()
        layers = []; in_dim = input_dim
        for i in range(num_hidden):
            out_dim = hidden_dim if i == 0 else hidden_dim // (2 ** i)
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(dropout_rate)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(dropout_rate * 0.8), nn.Linear(in_dim, 1))
    def forward(self, xf, xr):
        return (self.head(self.mlp(xf)) - self.head(self.mlp(xr))) / 2

class FullModelCrossAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.pooling = CrossEmbeddingAttention()
        self.ddg_predictor = DDGPredictor()
        self.ilddt_predictor = DDGPredictor()
    def forward(self, xf, xr):
        xfp = self.pooling(xf); xrp = self.pooling(xr)
        return (self.ddg_predictor(xfp, xrp).squeeze(-1),
                self.ilddt_predictor(xfp, xrp).squeeze(-1))

class DdgDataset:
    def __init__(self, xf, xr, y, ilddt):
        self.xf = np.asarray(xf, dtype=np.float32); self.xr = np.asarray(xr, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.float32); self.ilddt = np.asarray(ilddt, dtype=np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.xf[idx], self.xr[idx], self.y[idx], self.ilddt[idx]

In [47]:
# --- cache + fold loading ---
def load_cache(cache_path):
    data = np.load(cache_path, allow_pickle=True)
    for k in ['Xf', 'Xr', 'y', 'ilddt', 'ids']:
        if k not in data: raise ValueError(f"Cache missing key: {k}")
    complex_names = data['ids']
    ids = np.array([str(s).split('_', 1)[1] if '_' in str(s) else str(s) for s in complex_names])
    print(f"Loaded {len(data['y'])} samples from cache")
    return data['Xf'], data['Xr'], data['y'], data['ilddt'], ids, complex_names

def load_predefined_folds(folds_dir, ids, n_folds=10):
    def codes(path):
        s = set()
        with open(path) as fh:
            for line in fh:
                r = line.strip()
                if r and '_' in r: s.add(r.split('_', 1)[1])
        return s
    folds = []
    for k in range(1, n_folds + 1):
        fd = os.path.join(folds_dir, f"fold_{k}")
        tr_c = codes(os.path.join(fd, "train_complex_ids.txt"))
        te_c = codes(os.path.join(fd, "test_complex_ids.txt"))
        tr = np.array([i for i, d in enumerate(ids) if d in tr_c])
        te = np.array([i for i, d in enumerate(ids) if d in te_c])
        print(f"  fold {k}: {len(tr)} train, {len(te)} test")
        folds.append((tr, te))
    return folds

In [48]:
# --- train + eval one fold ---
def train_and_evaluate_fold(cfg, device, Xf_tr_all, Xr_tr_all, y_tr_all, il_tr_all,
                            Xf_te, Xr_te, y_te, il_te):
    Xf_tr, Xf_val, Xr_tr, Xr_val, y_tr, y_val, il_tr, il_val = train_test_split(
        Xf_tr_all, Xr_tr_all, y_tr_all, il_tr_all,
        test_size=0.10, random_state=cfg['random_state'])
    tl = DataLoader(DdgDataset(Xf_tr, Xr_tr, y_tr, il_tr), batch_size=cfg['batch_size'], shuffle=True)
    vl = DataLoader(DdgDataset(Xf_val, Xr_val, y_val, il_val), batch_size=cfg['batch_size'], shuffle=False)
    te = DataLoader(DdgDataset(Xf_te, Xr_te, y_te, il_te), batch_size=cfg['batch_size'], shuffle=False)

    model = FullModelCrossAttention().to(device)
    opt = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    mse = nn.MSELoss()
    lf = lambda pd_, pi_, td_, ti_: 1.0 * mse(pd_, td_) + 0.2 * mse(pi_, ti_)

    best_val, best_state = float('inf'), None
    for _ in range(1, cfg['epochs'] + 1):
        model.train()
        for xf, xr, yy, il in tl:
            xf, xr, yy, il = xf.to(device), xr.to(device), yy.to(device), il.to(device)
            loss = lf(*model(xf, xr), yy, il)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); vls = []
        with torch.no_grad():
            for xf, xr, yy, il in vl:
                xf, xr, yy, il = xf.to(device), xr.to(device), yy.to(device), il.to(device)
                vls.append(lf(*model(xf, xr), yy, il).item())
        mv = float(np.mean(vls))
        if mv < best_val:
            best_val = mv; best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)

    yt, yp, it, ip = [], [], [], []
    model.eval()
    with torch.no_grad():
        for xf, xr, yy, il in te:
            xf, xr = xf.to(device), xr.to(device)
            pd_, pi_ = model(xf, xr)
            yp.extend(pd_.cpu().numpy()); yt.extend(yy.numpy())
            ip.extend(pi_.cpu().numpy()); it.extend(il.numpy())
    return (np.array(yp), np.array(yt), np.array(ip), np.array(it),
            pearsonr(yt, yp)[0], spearmanr(yt, yp)[0],
            pearsonr(it, ip)[0], spearmanr(it, ip)[0], model)

In [49]:
# --- run one split (all 10 folds) ---
def run_cv(cfg, folds_dir, model_save_dir, output_csv):
    torch.manual_seed(cfg['random_state']); np.random.seed(cfg['random_state'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('device:', device, '| folds_dir:', folds_dir)
    os.makedirs(model_save_dir, exist_ok=True)
    Xf, Xr, y, ilddt, ids, names = load_cache(cfg['cache'])
    folds = load_predefined_folds(folds_dir, ids, cfg['n_folds'])

    A = dict(pd=[], td=[], pi=[], ti=[], names=[], fold=[]); fc = []
    for fi, (tr, te) in enumerate(folds):
        print(f"\n=== fold {fi+1}/{cfg['n_folds']}  train {len(tr)} test {len(te)} ===")
        (pd_, td_, pi_, ti_, ped, spd, pei, spi, model) = train_and_evaluate_fold(
            cfg, device, Xf[tr], Xr[tr], y[tr], ilddt[tr], Xf[te], Xr[te], y[te], ilddt[te])
        torch.save({'model_state_dict': model.state_dict()},
                   os.path.join(model_save_dir, f'fold_{fi}_checkpoint.pth'))
        A['pd'].extend(pd_); A['td'].extend(td_); A['pi'].extend(pi_); A['ti'].extend(ti_)
        A['names'].extend(names[te]); A['fold'].extend([fi]*len(te))
        fc.append({'fold': fi, 'pearson_ddg': float(ped), 'spearman_ddg': float(spd),
                   'pearson_ilddt': float(pei), 'spearman_ilddt': float(spi)})
        print(f"  ddG P {ped:.4f} S {spd:.4f} | ilddt P {pei:.4f} S {spi:.4f}")
    opd, osd = pearsonr(A['td'], A['pd'])[0], spearmanr(A['td'], A['pd'])[0]
    opi, osi = pearsonr(A['ti'], A['pi'])[0], spearmanr(A['ti'], A['pi'])[0]
    print(f"\n=== POOLED  ddG P {opd:.4f} S {osd:.4f} | ilddt P {opi:.4f} S {osi:.4f} ===")
    pd.DataFrame({'complex_name': A['names'], 'predicted_ddG': A['pd'], 'true_ddG': A['td'],
                  'predicted_ilddt': A['pi'], 'true_ilddt': A['ti'], 'fold': A['fold']}
                 ).to_csv(output_csv, index=False)
    summary = {'overall_pearson_ddg': float(opd), 'overall_spearman_ddg': float(osd),
               'overall_pearson_ilddt': float(opi), 'overall_spearman_ilddt': float(osi),
               'fold_results': fc, 'folds_dir': folds_dir, 'model_save_dir': model_save_dir}
    with open(os.path.splitext(output_csv)[0] + '_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    return summary

In [50]:
# --- CONFIG: edit these paths, then run the cell below ---
CONFIG = {
    'cache':"/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/ProtBFF/model_benchmarking/score_caches/skempi_score_cache.npz",
    'lr': 1e-4, 'weight_decay': 1e-5, 'epochs': 50,
    'batch_size': 32, 'n_folds': 10, 'random_state': 42,
}
CV_BASE = '/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_out/cv_structural'
OUT_BASE = 'model_out'
TMS = [50, 70, 90]

In [8]:
# --- run all TM splits (train + save checkpoints + pooled metrics) ---
summaries = {}
for T in TMS:
    folds_dir = f"{CV_BASE}/tm{T}"
    model_save_dir = f"{OUT_BASE}/skempi_prosst_tm{T}"
    output_csv = f"results_tm{T}.csv"
    print("\n" + "="*70 + f"\nTM {T/100:.1f}\n" + "="*70)
    summaries[T] = run_cv(CONFIG, folds_dir, model_save_dir, output_csv)

print("\n==== SUMMARY (pooled ddG) ====")
for T, s in summaries.items():
    print(f"TM 0.{T}:  Pearson {s['overall_pearson_ddg']:.4f}  Spearman {s['overall_spearman_ddg']:.4f}")


TM 0.5
device: cuda | folds_dir: /n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_out/cv_structural/tm50
Loaded 6631 samples from cache
  fold 1: 2565 train, 4066 test
  fold 2: 5709 train, 922 test
  fold 3: 6244 train, 387 test
  fold 4: 6386 train, 245 test
  fold 5: 6462 train, 169 test
  fold 6: 6463 train, 168 test
  fold 7: 6463 train, 168 test
  fold 8: 6461 train, 170 test
  fold 9: 6463 train, 168 test
  fold 10: 6463 train, 168 test

=== fold 1/10  train 2565 test 4066 ===
  ddG P 0.1771 S 0.1836 | ilddt P 0.0210 S 0.1435

=== fold 2/10  train 5709 test 922 ===
  ddG P 0.4888 S 0.4908 | ilddt P 0.1146 S 0.4240

=== fold 3/10  train 6244 test 387 ===
  ddG P 0.4197 S 0.4406 | ilddt P 0.0611 S 0.5901

=== fold 4/10  train 6386 test 245 ===
  ddG P 0.1271 S 0.1851 | ilddt P -0.0501 S 0.1541

=== fold 5/10  train 6462 test 169 ===
  ddG P 0.5804 S 0.4993 | ilddt P 0.1665 S 0.1739

=== fold 6/10  train 6463 test 168 ===
  ddG 

In [4]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr
for T in [50,70,90]:
    df = pd.read_csv(f"results_tm{T}.csv")
    df['cx'] = df['complex_name'].astype(str).str.split('_',n=1).str[1]
    rs=[]
    for cx,g in df.groupby('cx'):
        if len(g)>=10 and g['true_ddG'].std()>1e-6 and g['predicted_ddG'].std()>1e-6:
            r=pearsonr(g['true_ddG'],g['predicted_ddG'])[0]
            if np.isfinite(r): rs.append(r)
    rs=np.array(rs)
    print(f"TM{T}: per-complex mean r={rs.mean():.3f} median={np.median(rs):.3f} n={len(rs)}")

TM50: per-complex mean r=0.244 median=0.275 n=120
TM70: per-complex mean r=0.238 median=0.289 n=120
TM90: per-complex mean r=0.261 median=0.292 n=120


In [7]:
print('run_cv' in dir(), 'CONFIG' in dir())

True True


In [ ]:
import subprocess
CSV = "/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/ProtBFF/data/SKEMPI2_filtered_final.csv"
CVB = "/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_out/cv_structural"

# build 3 shuffled replicates matching tm90 fold sizes (different seeds)
for s in [0,1,2]:
    subprocess.run(["python","/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/shuffle_control.py","--csv",CSV,
                    "--ref_folds",f"{CVB}/tm90","--out",f"{CVB}/tm90_shuffled_s{s}",
                    "--seed",str(s)], check=True)

# retrain on each shuffled control (same trainer, same cache)
from scipy.stats import pearsonr
import pandas as pd
for s in [0,1,2]:
    run_cv(CONFIG, folds_dir=f"{CVB}/tm90_shuffled_s{s}",
           model_save_dir=f"model_out/shuf_s{s}", output_csv=f"shuf_s{s}.csv")

# compare pooled, excluding the big fold-1 in BOTH so geometry matches
def pooled_nofold1(csv):
    df=pd.read_csv(csv); d=df[df['fold']!=0]
    return pearsonr(d.true_ddG,d.predicted_ddG)[0]
print("\n=== leakage test (pooled, excl. fold 1) ===")
print("structure  tm90:", round(pooled_nofold1("results_tm90.csv"),3))
for s in [0,1,2]:
    print(f"shuffled s{s}:", round(pooled_nofold1(f"shuf_s{s}.csv"),3))

target per-fold test rows: [3543, 456, 387, 350, 325, 315, 315, 314, 313, 313]
achieved per-fold test rows: [3473, 614, 349, 324, 262, 244, 323, 508, 268, 266]
wrote shuffled-control folds -> /n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_out/cv_structural/tm90_shuffled_s0
target per-fold test rows: [3543, 456, 387, 350, 325, 315, 315, 314, 313, 313]
achieved per-fold test rows: [3516, 430, 391, 391, 312, 350, 287, 299, 308, 347]
wrote shuffled-control folds -> /n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_out/cv_structural/tm90_shuffled_s1
target per-fold test rows: [3543, 456, 387, 350, 325, 315, 315, 314, 313, 313]
achieved per-fold test rows: [3528, 438, 354, 307, 304, 273, 304, 491, 280, 352]
wrote shuffled-control folds -> /n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_out/cv_structural/tm90_shuffled_s2
device: cuda 

In [7]:
run_cv(CONFIG,
       folds_dir="/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_mva/cv/60_percent",
       model_save_dir="model_out/skempi_prosst_mva60",
       output_csv="results_mva60.csv")

device: cuda | folds_dir: /n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff/Protbff_updates/clustering_fixes/split_mva/cv/60_percent
Loaded 6631 samples from cache
  fold 1: 4676 train, 1955 test
  fold 2: 5482 train, 1149 test
  fold 3: 6175 train, 456 test
  fold 4: 6192 train, 439 test
  fold 5: 6192 train, 439 test
  fold 6: 6192 train, 439 test
  fold 7: 6192 train, 439 test
  fold 8: 6192 train, 439 test
  fold 9: 6193 train, 438 test
  fold 10: 6193 train, 438 test

=== fold 1/10  train 4676 test 1955 ===
  ddG P 0.1990 S 0.2709 | ilddt P 0.0451 S 0.2391

=== fold 2/10  train 5482 test 1149 ===
  ddG P 0.3230 S 0.2244 | ilddt P -0.0655 S 0.1845

=== fold 3/10  train 6175 test 456 ===
  ddG P 0.5619 S 0.5049 | ilddt P -0.0105 S 0.2726

=== fold 4/10  train 6192 test 439 ===
  ddG P 0.3895 S 0.4147 | ilddt P 0.0555 S 0.4574

=== fold 5/10  train 6192 test 439 ===
  ddG P 0.3526 S 0.3659 | ilddt P 0.0073 S 0.2556

=== fold 6/10  train 6192 test 439 ===
  ddG P 0.4109 S

{'overall_pearson_ddg': 0.2942509651184082,
 'overall_spearman_ddg': 0.31382318662592207,
 'overall_pearson_ilddt': 0.02080516889691353,
 'overall_spearman_ilddt': 0.29451960716724557,
 'fold_results': [{'fold': 0,
   'pearson_ddg': 0.19895949959754944,
   'spearman_ddg': 0.27089349504428806,
   'pearson_ilddt': 0.0450628288090229,
   'spearman_ilddt': 0.23910908052164648},
  {'fold': 1,
   'pearson_ddg': 0.3229960799217224,
   'spearman_ddg': 0.224428157265074,
   'pearson_ilddt': -0.06552230566740036,
   'spearman_ilddt': 0.18447810352271227},
  {'fold': 2,
   'pearson_ddg': 0.5619257092475891,
   'spearman_ddg': 0.5049146887111591,
   'pearson_ilddt': -0.010450050234794617,
   'spearman_ilddt': 0.27257650217222923},
  {'fold': 3,
   'pearson_ddg': 0.38945919275283813,
   'spearman_ddg': 0.41465433341454483,
   'pearson_ilddt': 0.055515654385089874,
   'spearman_ilddt': 0.45741936611404527},
  {'fold': 4,
   'pearson_ddg': 0.3525765538215637,
   'spearman_ddg': 0.36587003581907585,
 

In [8]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr, spearmanr
df = pd.read_csv("results_mva60.csv")

# 1. mean of per-fold correlations (equal weight per fold)
fr = [pearsonr(g.true_ddG, g.predicted_ddG)[0] for _,g in df.groupby('fold')]
print("mean per-fold r:", round(np.mean(fr),3))

# 2. z-scored within fold, then pooled (removes between-model offset/scale)
d = df.copy()
d['pz'] = d.groupby('fold')['predicted_ddG'].transform(lambda x:(x-x.mean())/x.std())
d['tz'] = d.groupby('fold')['true_ddG'].transform(lambda x:(x-x.mean())/x.std())
print("pooled (fold z-scored):", round(pearsonr(d.tz,d.pz)[0],3))

# 3. per-complex (fold-size blind)
d['cx']=d['complex_name'].astype(str).str.split('_',n=1).str[1]
rs=[pearsonr(g.true_ddG,g.predicted_ddG)[0] for _,g in d.groupby('cx')
    if len(g)>=10 and g.true_ddG.std()>1e-6 and g.predicted_ddG.std()>1e-6]
rs=[r for r in rs if np.isfinite(r)]
print("per-complex mean r:", round(np.mean(rs),3), "median", round(np.median(rs),3), "n",len(rs))

mean per-fold r: 0.401
pooled (fold z-scored): 0.347
per-complex mean r: 0.261 median 0.288 n 120


In [1]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr

a = pd.read_csv("results_tm90.csv")
b = pd.read_csv("results_mva60.csv")
m = a.merge(b, on='complex_name', suffixes=('_tm','_mva'))
print("rows matched:", len(m), "of", len(a), len(b))

# sanity: same ground truth?
print("true_ddG identical:", np.allclose(m.true_ddG_tm, m.true_ddG_mva))

# THE test: how correlated are the two runs' predictions?
r = pearsonr(m.predicted_ddG_tm, m.predicted_ddG_mva)[0]
print("pred-vs-pred r:", round(r,4))
print("mean |diff|:", round(np.abs(m.predicted_ddG_tm - m.predicted_ddG_mva).mean(),4))

# did fold membership actually change?
same_fold = (m.fold_tm == m.fold_mva).mean()
print("fraction of entries in the same fold number:", round(same_fold,3))
print(pd.crosstab(m.fold_tm, m.fold_mva).to_string())

rows matched: 6631 of 6631 6631
true_ddG identical: True
pred-vs-pred r: 0.7816
mean |diff|: 0.5375
fraction of entries in the same fold number: 0.331
fold_mva     0     1    2    3    4    5    6    7    8    9
fold_tm                                                     
0         1955  1149    0   10   19   26   31  151  142   60
1            0     0  456    0    0    0    0    0    0    0
2            0     0    0  387    0    0    0    0    0    0
3            0     0    0    0  350    0    0    0    0    0
4            0     0    0    0    0  325    0    0    0    0
5            0     0    0    3   23   12  253   11    3   10
6            0     0    0    0    3   51   29   41  142   49
7            0     0    0   12   10   13   23  105   10  141
8            0     0    0   21    8    7   53   21   59  144
9            0     0    0    6   26    5   50  110   82   34


In [51]:
import copy

In [52]:
# ============================================================
# CELL 2 — parameterized model (defaults == original paper model)
# ============================================================
class CrossEmbeddingAttention(nn.Module):
    def __init__(self, reduced_dim=512, num_scores=5, embed_dim=768,
                 proj_dropout=0.4, attn_dropout=0.4, num_heads=8):
        super().__init__()
        self.embed_dim=embed_dim; self.num_scores=num_scores; self.reduced_dim=reduced_dim
        self.embedding_projections = nn.ModuleList([
            nn.Sequential(nn.Linear(embed_dim, reduced_dim), nn.ReLU(), nn.Dropout(proj_dropout))
            for _ in range(num_scores)])
        self.cross_attention = nn.MultiheadAttention(embed_dim=reduced_dim, num_heads=num_heads,
                                                     dropout=0.3, batch_first=True)
        self.attention_norm = nn.LayerNorm(reduced_dim)
        self.final_attention = nn.Sequential(nn.Linear(num_scores, 32), nn.ReLU(),
                                             nn.Dropout(attn_dropout), nn.Linear(32, num_scores))
    def forward(self, x):
        b=x.size(0)
        emb=x.view(b, self.num_scores, self.embed_dim)
        proj=torch.stack([p(emb[:,i,:]) for i,p in enumerate(self.embedding_projections)],dim=1)
        att,_=self.cross_attention(proj,proj,proj)
        att=self.attention_norm(att+proj)
        att_t=att.transpose(1,2)
        w=F.softmax(self.final_attention(att_t),dim=-1)
        return (w*att_t).sum(dim=-1)

class DDGPredictor(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=768, dropout_rate=0.35, num_hidden=3):
        super().__init__()
        layers=[]; in_dim=input_dim
        for i in range(num_hidden):
            out_dim = hidden_dim if i==0 else hidden_dim//(2**i)
            layers += [nn.Linear(in_dim,out_dim), nn.ReLU(), nn.Dropout(dropout_rate)]
            in_dim=out_dim
        self.mlp=nn.Sequential(*layers)
        self.head=nn.Sequential(nn.Dropout(dropout_rate*0.8), nn.Linear(in_dim,1))
    def forward(self, xf, xr):
        return (self.head(self.mlp(xf)) - self.head(self.mlp(xr)))/2

class FullModelCrossAttention(nn.Module):
    def __init__(self, cfg=None):
        super().__init__()
        c = cfg or {}
        self.pooling = CrossEmbeddingAttention(
            reduced_dim=c.get('reduced_dim',512), embed_dim=c.get('embed_dim',768),
            proj_dropout=c.get('proj_dropout',0.4), attn_dropout=c.get('attn_dropout',0.4),
            num_heads=c.get('num_heads',8))
        mk = lambda: DDGPredictor(input_dim=c.get('reduced_dim',512),
                                  hidden_dim=c.get('hidden_dim',768),
                                  dropout_rate=c.get('mlp_dropout',0.35),
                                  num_hidden=c.get('num_hidden',3))
        self.ddg_predictor=mk(); self.ilddt_predictor=mk()
    def forward(self, xf, xr):
        xfp=self.pooling(xf); xrp=self.pooling(xr)
        return (self.ddg_predictor(xfp,xrp).squeeze(-1),
                self.ilddt_predictor(xfp,xrp).squeeze(-1))

class DdgDataset:
    def __init__(self, xf, xr, y, ilddt):
        self.xf=np.asarray(xf,dtype=np.float32); self.xr=np.asarray(xr,dtype=np.float32)
        self.y=np.asarray(y,dtype=np.float32);   self.ilddt=np.asarray(ilddt,dtype=np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.xf[i], self.xr[i], self.y[i], self.ilddt[i]

In [53]:
# ============================================================
# CELL 3 — data, folds, and CLUSTER-AWARE validation split
# ============================================================
def load_cache(cache_path):
    d=np.load(cache_path, allow_pickle=True)
    names=d['ids']
    ids=np.array([str(s).split('_',1)[1] if '_' in str(s) else str(s) for s in names])
    print(f"cache: {len(d['y'])} samples")
    return d['Xf'], d['Xr'], d['y'], d['ilddt'], ids, names

def load_cluster_map(clusters_tsv):
    """complex -> cluster_id (from 04_cluster_report.py)."""
    df=pd.read_csv(clusters_tsv, sep='\t')
    return dict(zip(df['complex'].astype(str), df['cluster_id']))

def load_predefined_folds(folds_dir, ids, n_folds=10):
    def codes(p):
        s=set()
        with open(p) as fh:
            for line in fh:
                r=line.strip()
                if r and '_' in r: s.add(r.split('_',1)[1])
        return s
    folds=[]
    for k in range(1,n_folds+1):
        fd=os.path.join(folds_dir,f"fold_{k}")
        tr_c=codes(os.path.join(fd,"train_complex_ids.txt"))
        te_c=codes(os.path.join(fd,"test_complex_ids.txt"))
        tr=np.array([i for i,d in enumerate(ids) if d in tr_c])
        te=np.array([i for i,d in enumerate(ids) if d in te_c])
        folds.append((tr,te))
    return folds

def cluster_aware_val_split(train_idx, ids, cluster_map, val_frac=0.10, seed=42):
    """
    Hold out WHOLE CLUSTERS from the training portion for validation, so the
    val set contains complexes (and homologs) the model never trains on.
    Falls back to complex-as-its-own-cluster for anything missing from the map.
    """
    rng=np.random.default_rng(seed)
    by_cluster={}
    for i in train_idx:
        cx=ids[i]
        cl=cluster_map.get(cx, f'__solo_{cx}')
        by_cluster.setdefault(cl,[]).append(i)
    clusters=list(by_cluster)
    rng.shuffle(clusters)
    target=int(round(val_frac*len(train_idx)))
    # add clusters toward target, skipping any that would overshoot badly
    val=[]
    for cl in clusters:
        if len(val) >= target: break
        if val and len(val)+len(by_cluster[cl]) > 1.5*target: continue
        val.extend(by_cluster[cl])
    if not val:   # every cluster is huge -> take the smallest one
        cl=min(clusters, key=lambda c: len(by_cluster[c]))
        val=list(by_cluster[cl])
    val=np.array(sorted(val))
    tr=np.array(sorted(set(train_idx)-set(val.tolist())))
    return tr, val

In [54]:
# ============================================================
# CELL 4 — train one fold; epoch chosen on VALIDATION only
# ============================================================
def train_fold(cfg, device, Xf, Xr, y, ilddt, tr_idx, val_idx, te_idx, return_model=False):
    tl=DataLoader(DdgDataset(Xf[tr_idx],Xr[tr_idx],y[tr_idx],ilddt[tr_idx]),
                  batch_size=cfg['batch_size'], shuffle=True)
    vl=DataLoader(DdgDataset(Xf[val_idx],Xr[val_idx],y[val_idx],ilddt[val_idx]),
                  batch_size=cfg['batch_size'], shuffle=False)
    te=DataLoader(DdgDataset(Xf[te_idx],Xr[te_idx],y[te_idx],ilddt[te_idx]),
                  batch_size=cfg['batch_size'], shuffle=False)

    model=FullModelCrossAttention(cfg).to(device)
    opt=optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    mse=nn.MSELoss(); w=cfg.get('ilddt_weight',0.2)
    lf=lambda pd_,pi_,td_,ti_: mse(pd_,td_) + w*mse(pi_,ti_)

    def evaluate(loader):
        model.eval(); yt,yp,it,ip=[],[],[],[]
        with torch.no_grad():
            for xf,xr,yy,il in loader:
                pd_,pi_=model(xf.to(device),xr.to(device))
                yp.extend(pd_.cpu().numpy()); yt.extend(yy.numpy())
                ip.extend(pi_.cpu().numpy()); it.extend(il.numpy())
        return np.array(yt),np.array(yp),np.array(it),np.array(ip)

    best_score=-np.inf; best_state=None; bad=0
    patience=cfg.get('patience', cfg['epochs'])
    for ep in range(1, cfg['epochs']+1):
        model.train()
        for xf,xr,yy,il in tl:
            xf,xr,yy,il=xf.to(device),xr.to(device),yy.to(device),il.to(device)
            loss=lf(*model(xf,xr),yy,il)
            opt.zero_grad(); loss.backward(); opt.step()
        vt,vp,_,_=evaluate(vl)
        score = pearsonr(vt,vp)[0] if (vt.std()>1e-8 and vp.std()>1e-8) else -np.inf
        if not np.isfinite(score): score=-np.inf
        if score>best_score:
            best_score=score; bad=0
            best_state={k:v.cpu() for k,v in model.state_dict().items()}
        else:
            bad+=1
            if bad>=patience: break
    if best_state is not None: model.load_state_dict(best_state)

    vt,vp,_,_=evaluate(vl)
    tt,tp,it,ip=evaluate(te)
    out=dict(val_pearson=float(pearsonr(vt,vp)[0]), val_spearman=float(spearmanr(vt,vp)[0]),
             test_pearson=float(pearsonr(tt,tp)[0]), test_spearman=float(spearmanr(tt,tp)[0]),
             test_pred=tp, test_true=tt, ilddt_pred=ip, ilddt_true=it)
    if return_model: out['model']=model
    return out

In [55]:
# ============================================================
# CELL 5 — full CV run (val-selected epochs, cluster-aware val)
# ============================================================
def run_cv_v2(cfg, folds_dir, clusters_tsv, model_save_dir=None, output_csv=None,
              folds_subset=None, verbose=True):
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    Xf,Xr,y,ilddt,ids,names=load_cache(cfg['cache'])
    cmap=load_cluster_map(clusters_tsv)
    folds=load_predefined_folds(folds_dir, ids, cfg['n_folds'])
    if folds_subset is not None:
        folds=[folds[i] for i in folds_subset]
    if model_save_dir: os.makedirs(model_save_dir, exist_ok=True)

    rows=[]; per_fold=[]
    for fi,(tr_all,te) in enumerate(folds):
        torch.manual_seed(cfg['random_state']+fi); np.random.seed(cfg['random_state']+fi)
        tr,val=cluster_aware_val_split(tr_all, ids, cmap,
                                       val_frac=cfg.get('val_frac',0.10),
                                       seed=cfg['random_state']+fi)
        r=train_fold(cfg, device, Xf,Xr,y,ilddt, tr,val,te,
                     return_model=bool(model_save_dir))
        if model_save_dir:
            torch.save({'model_state_dict': r['model'].state_dict(), 'cfg': cfg},
                       os.path.join(model_save_dir, f'fold_{fi}_checkpoint.pth'))
        per_fold.append({k:r[k] for k in
                         ('val_pearson','val_spearman','test_pearson','test_spearman')})
        if verbose:
            print(f"  fold {fi+1}: train {len(tr)} val {len(val)} test {len(te)} | "
                  f"VAL r={r['val_pearson']:.3f} | test r={r['test_pearson']:.3f}")
        if output_csv:
            rows.append(pd.DataFrame({'complex_name':names[te],'fold':fi,
                                      'true_ddG':r['test_true'],'predicted_ddG':r['test_pred'],
                                      'true_ilddt':r['ilddt_true'],'predicted_ilddt':r['ilddt_pred']}))
    summary=dict(
        mean_val_pearson=float(np.mean([f['val_pearson'] for f in per_fold])),
        mean_test_pearson=float(np.mean([f['test_pearson'] for f in per_fold])),
        mean_test_spearman=float(np.mean([f['test_spearman'] for f in per_fold])),
        per_fold=per_fold, folds_dir=folds_dir, cfg={k:v for k,v in cfg.items()})
    if output_csv and rows:
        df=pd.concat(rows, ignore_index=True); df.to_csv(output_csv, index=False)
        summary['pooled_test_pearson']=float(pearsonr(df.true_ddG,df.predicted_ddG)[0])
        with open(os.path.splitext(output_csv)[0]+'_summary.json','w') as f:
            json.dump(summary,f,indent=2)
    return summary

In [56]:
# ============================================================
# CELL 6 — hyperparameter sweep, SELECTED ON VALIDATION
# ============================================================
def sweep(base_cfg, grid, folds_dir, clusters_tsv, folds_subset=(1,2,3)):
    """
    grid: list of dicts of overrides. Selection uses mean VAL Pearson only.
    Test numbers are printed for transparency but must NOT drive the choice.
    """
    results=[]
    for gi,ov in enumerate(grid):
        cfg=copy.deepcopy(base_cfg); cfg.update(ov)
        print(f"\n[{gi+1}/{len(grid)}] {ov}")
        s=run_cv_v2(cfg, folds_dir, clusters_tsv, folds_subset=list(folds_subset), verbose=False)
        results.append(dict(overrides=json.dumps(ov),
                            mean_val_pearson=s['mean_val_pearson'],
                            mean_test_pearson=s['mean_test_pearson']))
        print(f"   VAL r={s['mean_val_pearson']:.4f}   (test r={s['mean_test_pearson']:.4f})")
    df=pd.DataFrame(results).sort_values('mean_val_pearson', ascending=False)
    print("\n=== ranked by VALIDATION Pearson (selection criterion) ===")
    print(df.to_string(index=False))
    return df

In [43]:
# ============================================================
# CELL 7 — CONFIG, paths, and the sweep grid
# ============================================================
BASE = '/n/netscratch/shakhnovich_lab/Lab/jonathanfeldman/test_protbff'
CONFIG = {
    'cache':  f'{BASE}/ProtBFF/model_benchmarking/score_caches/skempi_score_cache.npz',
    'lr': 1e-4, 'weight_decay': 1e-5, 'epochs': 50, 'batch_size': 32,
    'n_folds': 10, 'random_state': 42, 'val_frac': 0.10,
    'ilddt_weight': 0.2, 'patience': 15,
    # architecture (defaults reproduce the paper model)
    'reduced_dim': 512, 'hidden_dim': 768, 'num_hidden': 3,
    'mlp_dropout': 0.35, 'proj_dropout': 0.4, 'attn_dropout': 0.4, 'num_heads': 8,
}
FOLDS    = f'{BASE}/Protbff_updates/clustering_fixes/split_mva/cv/60_percent'
CLUSTERS = f'{BASE}/Protbff_updates/clustering_fixes/split_mva/clusters.tsv'

GRID = [
    {'ilddt_weight': 0.3, 'lr': 5e-5},                         # drop the dead auxiliary head
]

In [44]:
# ============================================================
# CELL 8 — run the sweep (3 folds, fast) and pick by VALIDATION
# ============================================================
res = sweep(CONFIG, GRID, FOLDS, CLUSTERS, folds_subset=(0, 0))
res.to_csv('sweep_results.csv', index=False)


[1/1] {'ilddt_weight': 0.3, 'lr': 5e-05}
cache: 6631 samples
   VAL r=0.5163   (test r=0.2182)

=== ranked by VALIDATION Pearson (selection criterion) ===
                         overrides  mean_val_pearson  mean_test_pearson
{"ilddt_weight": 0.3, "lr": 5e-05}          0.516257            0.21824


In [38]:
# ============================================================
# CELL 9 — full 10-fold with the VAL-selected winner
# ============================================================
import json as _json
best = _json.loads(res.iloc[0]['overrides'])       # top row = best VAL Pearson
print('winning overrides:', best)
cfg = {**CONFIG, **best}

s = run_cv_v2(cfg, FOLDS, CLUSTERS,
              model_save_dir='model_out/mva60_tuned',
              output_csv='results_mva60_tuned.csv')
print(f"\nmean VAL  r = {s['mean_val_pearson']:.4f}")
print(f"mean TEST r = {s['mean_test_pearson']:.4f}   "
      f"(pooled {s.get('pooled_test_pearson', float('nan')):.4f})")

winning overrides: {'ilddt_weight': 0.25, 'lr': 0.0002}
cache: 6631 samples
  fold 1: train 4202 val 474 test 1955 | VAL r=0.512 | test r=0.240
  fold 2: train 4917 val 565 test 1149 | VAL r=0.276 | test r=0.368
  fold 3: train 5475 val 700 test 456 | VAL r=0.432 | test r=0.500
  fold 4: train 5488 val 704 test 439 | VAL r=0.487 | test r=0.417
  fold 5: train 5475 val 717 test 439 | VAL r=0.449 | test r=0.251
  fold 6: train 5549 val 643 test 439 | VAL r=0.389 | test r=0.337
  fold 7: train 5562 val 630 test 439 | VAL r=0.466 | test r=0.422
  fold 8: train 5365 val 827 test 439 | VAL r=0.496 | test r=0.367
  fold 9: train 5299 val 894 test 438 | VAL r=0.512 | test r=0.293
  fold 10: train 5567 val 626 test 438 | VAL r=0.496 | test r=0.449

mean VAL  r = 0.4516
mean TEST r = 0.3645   (pooled 0.3012)


In [57]:
# ============================================================
# CELL 10 — baseline for comparison: same protocol, paper settings
# (run this so the tuned number is compared like-for-like)
# ============================================================
s0 = run_cv_v2(CONFIG, FOLDS, CLUSTERS,
               model_save_dir='model_out/mva60_baseline',
               output_csv='results_mva60_baseline.csv')
print(f"baseline  mean VAL r = {s0['mean_val_pearson']:.4f}  "
      f"mean TEST r = {s0['mean_test_pearson']:.4f}")

cache: 6631 samples
  fold 1: train 4202 val 474 test 1955 | VAL r=0.525 | test r=0.216
  fold 2: train 4917 val 565 test 1149 | VAL r=0.289 | test r=0.303
  fold 3: train 5475 val 700 test 456 | VAL r=0.409 | test r=0.599
  fold 4: train 5488 val 704 test 439 | VAL r=0.484 | test r=0.348
  fold 5: train 5475 val 717 test 439 | VAL r=0.467 | test r=0.344
  fold 6: train 5549 val 643 test 439 | VAL r=0.418 | test r=0.453
  fold 7: train 5562 val 630 test 439 | VAL r=0.464 | test r=0.544
  fold 8: train 5365 val 827 test 439 | VAL r=0.504 | test r=0.336
  fold 9: train 5299 val 894 test 438 | VAL r=0.515 | test r=0.314
  fold 10: train 5567 val 626 test 438 | VAL r=0.478 | test r=0.403
baseline  mean VAL r = 0.4554  mean TEST r = 0.3861


In [58]:
# ============================================================
# CELL A1 — block transforms + num_scores-aware model patch
# Paste AFTER the existing cells of protbff_tuning.ipynb
# ============================================================
SCORE_NAMES = ['interface', 'burial', 'dihedral', 'SASA', 'lDDT']   # VERIFY vs merge_scores.py

def transform_blocks(X, mode, k=None, num_scores=5, dim=768):
    """Cache layout is (N, num_scores*dim); block k = embedding scaled by score k."""
    B = X.reshape(len(X), num_scores, dim)
    if mode == 'full':
        return X, num_scores
    if mode == 'mean':      # removes score DIVERSITY, keeps magnitude/information
        M = B.mean(axis=1, keepdims=True)
        return np.repeat(M, num_scores, axis=1).reshape(len(X), -1), num_scores
    if mode == 'single':    # one score, replicated into all 5 slots
        return np.repeat(B[:, k:k+1, :], num_scores, axis=1).reshape(len(X), -1), num_scores
    if mode == 'drop':      # leave-one-score-out
        idx = [i for i in range(num_scores) if i != k]
        return B[:, idx, :].reshape(len(X), -1), num_scores - 1
    if mode == 'onlyone':   # single score, model sees ONE block
        return B[:, k, :].reshape(len(X), -1), 1
    raise ValueError(mode)

# model must honour cfg['num_scores']
_orig_init = FullModelCrossAttention.__init__
def _patched_init(self, cfg=None):
    c = cfg or {}
    nn.Module.__init__(self)
    self.pooling = CrossEmbeddingAttention(
        reduced_dim=c.get('reduced_dim',512), num_scores=c.get('num_scores',5),
        embed_dim=c.get('embed_dim',768), proj_dropout=c.get('proj_dropout',0.4),
        attn_dropout=c.get('attn_dropout',0.4), num_heads=c.get('num_heads',8))
    mk = lambda: DDGPredictor(input_dim=c.get('reduced_dim',512),
                              hidden_dim=c.get('hidden_dim',768),
                              dropout_rate=c.get('mlp_dropout',0.35),
                              num_hidden=c.get('num_hidden',3))
    self.ddg_predictor = mk(); self.ilddt_predictor = mk()
FullModelCrossAttention.__init__ = _patched_init

# ============================================================
# CELL A2 — CV runner that applies a block transform
# ============================================================
def run_ablation(cfg, folds_dir, clusters_tsv, mode, k=None, verbose=False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    Xf, Xr, y, ilddt, ids, names = load_cache(cfg['cache'])
    Xf2, ns = transform_blocks(Xf, mode, k)
    Xr2, _  = transform_blocks(Xr, mode, k)
    cfg = {**cfg, 'num_scores': ns}
    cmap = load_cluster_map(clusters_tsv)
    folds = load_predefined_folds(folds_dir, ids, cfg['n_folds'])

    per_fold = []
    for fi, (tr_all, te) in enumerate(folds):
        torch.manual_seed(cfg['random_state']+fi); np.random.seed(cfg['random_state']+fi)
        tr, val = cluster_aware_val_split(tr_all, ids, cmap,
                                          val_frac=cfg.get('val_frac',0.10),
                                          seed=cfg['random_state']+fi)
        r = train_fold(cfg, device, Xf2, Xr2, y, ilddt, tr, val, te)
        per_fold.append(r['test_pearson'])
        if verbose: print(f"    fold {fi+1}: test r={r['test_pearson']:.3f}")
    return float(np.mean(per_fold)), float(np.std(per_fold)), per_fold

# ============================================================
# CELL A3 — run the ablations (paper hyperparameters)
# ============================================================
ABL_CFG = {**CONFIG}          # keep ilddt_weight 0.2 = paper settings

runs = [('full', None, 'full model (5 scores)'),
        ('mean', None, 'all blocks -> their MEAN (no score diversity)')]
runs += [('single', k, f'only {SCORE_NAMES[k]} (replicated)') for k in range(5)]
runs += [('drop',   k, f'drop {SCORE_NAMES[k]}')             for k in range(5)]

results = []
for mode, k, label in runs:
    m, s, pf = run_ablation(ABL_CFG, FOLDS, CLUSTERS, mode, k)
    results.append({'ablation': label, 'mean_test_r': m, 'std': s})
    print(f"{label:45s}  mean test r = {m:.4f}  (sd {s:.3f})")

abl = pd.DataFrame(results).sort_values('mean_test_r', ascending=False)
abl.to_csv('ablation_results.csv', index=False)
print(); print(abl.to_string(index=False))

cache: 6631 samples
full model (5 scores)                          mean test r = 0.3861  (sd 0.110)
cache: 6631 samples
all blocks -> their MEAN (no score diversity)  mean test r = 0.2883  (sd 0.086)
cache: 6631 samples
only interface (replicated)                    mean test r = 0.3028  (sd 0.096)
cache: 6631 samples
only burial (replicated)                       mean test r = 0.3368  (sd 0.087)
cache: 6631 samples
only dihedral (replicated)                     mean test r = 0.3114  (sd 0.072)
cache: 6631 samples
only SASA (replicated)                         mean test r = 0.1954  (sd 0.079)
cache: 6631 samples
only lDDT (replicated)                         mean test r = 0.2633  (sd 0.165)
cache: 6631 samples
drop interface                                 mean test r = 0.3523  (sd 0.094)
cache: 6631 samples
drop burial                                    mean test r = 0.3887  (sd 0.089)
cache: 6631 samples
drop dihedral                                  mean test r = 0.3588  (sd 0.109)
